# Classificação & Segmentação – Modelo de Coral-Sol 

Autor:  Viviane da Rosa Sommer

Atualizado: 12/09/2025

Objetivo: Esse código faz a triagem e segmentação automática de imagens para detecção de corais. Ele percorre pastas de imagens, utiliza um modelo YOLO para classificar cada imagem como "positiva" (presença de coral) ou "negativa" (ausência de coral) e, se detectar coral, aplica um segundo modelo para segmentar a região do coral na imagem. As imagens são então copiadas para pastas separadas ("positivo" ou "negativo"), junto com arquivos JSON contendo metadados da classificação e, para casos positivos, salva também a máscara segmentada do coral em formato PNG e NPZ. 

## Importações necessárias

In [ ]:
import numpy as np
import cv2
import os
from ultralytics import YOLO
import shutil
import torchvision
import torch
import json

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
INPUT_DIRECTORY_IMAGES = "fase1-0-7-3/validas"
OUTPUT_DIRECTORY = "fase2-0-7-3"
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

model_v1 = YOLO('pesos/best.pt')
model_v2 = YOLO('pesos/coral_yolov11_segm_2k_V6.1_parcial.pt')

## Funções necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray | None:
    """
    Reads an image from file, converts it to RGB, and crops the central vertical region.
    
    Args:
        filename (str): Path to the image file.
    
    Returns:
        np.ndarray | None: Cropped RGB image as uint8 or None if reading fails.
    """
    image = cv2.imread(filename)
    if image is None:
        return None
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width, _ = image.shape
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_image = image[top:bottom, :]
    return cropped_image.astype(np.uint8)

def process_results(results: list) -> object | None:
    """
    Returns the first result with non-None masks.
    
    Args:
        results (list): List of result objects.
    
    Returns:
        object | None: First result with masks or None.
    """
    for result in results:
        if getattr(result, "masks", None) is not None:
            return result
    return None

def prediction_coral(img: np.ndarray, model: object) -> np.ndarray:
    """
    Runs coral prediction on an image and returns the final coral mask.
    
    Args:
        img (np.ndarray): Input RGB image.
        model (object): Model object with callable interface.
    
    Returns:
        np.ndarray: Coral mask resized to original image shape.
    """
    original_height, original_width = img.shape[:2]
    coral_results = model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    if coral_best_result and getattr(coral_best_result, "masks", None) is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]
        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.1))[0]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]
        if len(coral_masks) > 0:
            nms_indices = torchvision.ops.nms(boxes[coral_indices, :4], coral_scores, iou_threshold=0.2)
            final_coral_mask = torch.any(coral_masks[nms_indices], dim=0).int().cpu().numpy() * 255
        else:
            final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    else:
        final_coral_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    return cv2.resize(final_coral_mask, (original_width, original_height), interpolation=cv2.INTER_NEAREST)

## Processamento das imagens por operação

In [ ]:
for operation_folder in os.listdir(INPUT_DIRECTORY_IMAGES):
    operation_path = os.path.join(INPUT_DIRECTORY_IMAGES, operation_folder)
    if not os.path.isdir(operation_path):
        continue
    output_operation_dir = os.path.join(OUTPUT_DIRECTORY, operation_folder)
    positive_path = os.path.join(output_operation_dir, "positivo")
    negative_path = os.path.join(output_operation_dir, "negativo")
    os.makedirs(positive_path, exist_ok=True)
    os.makedirs(negative_path, exist_ok=True)
    for filename in os.listdir(operation_path):
        file_path = os.path.join(operation_path, filename)
        if not os.path.isfile(file_path):
            continue
        if not filename.lower().endswith(IMAGE_EXTENSIONS):
            continue
        image = read_image(file_path)
        if image is None:
            print(f"Erro ao ler a imagem: {file_path}")
            continue
        try:
            results_v1 = model_v1.predict(file_path, verbose=False)
            if not hasattr(results_v1[0], "probs") or results_v1[0].probs is None:
                print(f"Sem probabilidades para: {file_path}")
                continue
            class_probabilities_v1 = results_v1[0].probs.data.cpu().numpy()
            predicted_label_index_v1 = np.argmax(class_probabilities_v1)
            v1_score = class_probabilities_v1[predicted_label_index_v1]
            predicted_label_v1 = 'Positive' if predicted_label_index_v1 == 1 else 'Negative'
            mask_v2 = None
            if predicted_label_v1 == 'Positive':
                mask_v2 = prediction_coral(image, model_v2)
                mask_filename = os.path.splitext(os.path.basename(file_path))[0] + "_mask.png"
                cv2.imwrite(os.path.join(positive_path, mask_filename), mask_v2)
            output_dir = positive_path if predicted_label_v1 == "Positive" else negative_path
            shutil.copy(file_path, output_dir)

            base_name = os.path.splitext(os.path.basename(file_path))[0]
            meta = {
                "nome_imagem": file_path,
                "probabilidade_classificacao": float(v1_score),
                "classe": predicted_label_v1
            }
            with open(os.path.join(output_dir, f"{base_name}.json"), "w") as f:
                json.dump(meta, f)

            if mask_v2 is not None:
                np.savez_compressed(os.path.join(positive_path, f"{base_name}_mask.npz"), mask=mask_v2)

        except Exception as e:
            print(f"Erro ao processar {file_path}: {e}")
            continue